In [1]:
# Sentiment Analysis using NLP Pipeline & ML Models

# ==============================
# 1. Import Libraries
# ==============================
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

nltk.download('stopwords')

# ==============================
# 2. Load Dataset
# ==============================
# Replace with your dataset path
df = pd.read_csv('Twitter_Data.csv')

print("Dataset Shape:", df.shape)
print(df.head())

# Make sure your dataset has columns: 'text' and 'label'

# ==============================
# 3. Data Understanding
# ==============================
print("\nClass Distribution:")
print(df['label'].value_counts())

# ==============================
# 4. Preprocessing
# ==============================
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [stemmer.stem(word) for word in words if word not in stop_words]
    return " ".join(words)

# Apply preprocessing
df['clean_text'] = df['text'].apply(preprocess)

print("\nSample Cleaned Text:")
print(df[['text', 'clean_text']].head())

# ==============================
# 5. Train-Test Split
# ==============================
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==============================
# 6. Feature Engineering
# ==============================

# Bag of Words
bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

# TF-IDF
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# ==============================
# 7. Model Training
# ==============================

models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier()
}

results = []

# Function to evaluate model
def evaluate_model(name, model, X_train, X_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    results.append([name, acc, prec, rec, f1])

# Evaluate all models on BoW
for name, model in models.items():
    evaluate_model(name + " (BoW)", model, X_train_bow, X_test_bow)

# Evaluate all models on TF-IDF
for name, model in models.items():
    evaluate_model(name + " (TF-IDF)", model, X_train_tfidf, X_test_tfidf)

# ==============================
# 8. Results
# ==============================
results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score"])

print("\nModel Comparison:")
print(results_df)

# ==============================
# 9. Insights (Write Your Own)
# ==============================
print("""
Insights:
1. TF-IDF usually performs better than BoW because it considers word importance.
2. Logistic Regression often gives the best balance between performance and speed.
3. Naive Bayes is fast but may be slightly less accurate.
4. Decision Trees can overfit on text data.

(Modify these insights based on your actual results)
""")

# ==============================
# 10. Save Results (Optional)
# ==============================
results_df.to_csv('model_results.csv', index=False)

print("\nPipeline Completed Successfully!")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\pavan\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


Dataset Shape: (162980, 2)
                                          clean_text  category
0  when modi promised “minimum government maximum...      -1.0
1  talk all the nonsense and continue all the dra...       0.0
2  what did just say vote for modi  welcome bjp t...       1.0
3  asking his supporters prefix chowkidar their n...       1.0
4  answer who among these the most powerful world...       1.0

Class Distribution:


KeyError: 'label'